# Week 6 — Day 4: Neural Networks and Frontier LLMs

Three different approaches to the price-prediction task:
1. **Human baseline** — predictions made by reading 100 test items and guessing
2. **PyTorch neural network** — 8-layer fully connected NN on bag-of-words features
3. **Frontier LLMs** — zero-shot prompting (no training): GPT-4.1 Nano, GPT-5.1, Claude Opus 4.5, Gemini 3 Pro, Gemini 2.5 Flash Lite, Grok 4.1 Fast

## Setup

In [ ]:
import os
import csv
import numpy as np
from dotenv import load_dotenv
from huggingface_hub import login
from litellm import completion
from tqdm.notebook import tqdm
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pricer.evaluator import evaluate
from pricer.items import Item

In [ ]:
LITE_MODE = False
load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
username = "YOUR_HF_USERNAME"  # replace with your Hugging Face username
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

## Human baseline

Write the first 100 test items to a CSV, fill in price guesses by hand, then read the predictions back and evaluate.

In [ ]:
# Write the test set to a CSV — fill in column 2 with your predicted prices
with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [ ]:
# Read predictions back from human_out.csv
human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [ ]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [ ]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")

In [ ]:
evaluate(human_pricer, test, size=100)

## PyTorch neural network

8-layer fully connected NN on a 5,000-dim binary bag-of-words. ReLU activations, MSE loss, Adam optimizer, 2 epochs.

This is a sneak preview — week 7 / 8 go deeper on neural network training.

In [ ]:
y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
# HashingVectorizer with binary=True gives one-hot bag-of-words vectors
np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [ ]:
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_tensor, y_train_tensor, test_size=0.01, random_state=42
)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_params:,}")

In [ ]:
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()
        # 4 stages of training: forward pass, loss, backward pass, optimize
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

In [ ]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [ ]:
evaluate(neural_network, test)

## Frontier LLMs (zero-shot)

No training — just prompting. The models leverage their world knowledge of product prices.

In [ ]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [ ]:
print(test[0].summary)
print()
messages_for(test[0])

### GPT-4.1 Nano

In [ ]:
def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
print("Predicted:", gpt_4__1_nano(test[0]))
print("Actual:", test[0].price)

In [ ]:
evaluate(gpt_4__1_nano, test)

### Claude Opus 4.5

In [ ]:
def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(claude_opus_4_5, test)

### Gemini 3 Pro

In [ ]:
def gemini_3_pro_preview(item):
    response = completion(
        model="gemini/gemini-3-pro-preview",
        messages=messages_for(item),
        reasoning_effort='low'
    )
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

### Gemini 2.5 Flash Lite

In [ ]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_2__5_flash_lite, test)

### Grok 4.1 Fast

In [ ]:
def grok_4__1_fast(item):
    response = completion(
        model="xai/grok-4-1-fast-non-reasoning",
        messages=messages_for(item),
        seed=42
    )
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

### GPT-5.1 (high reasoning effort)

In [ ]:
def gpt_5__1(item):
    response = completion(
        model="gpt-5.1",
        messages=messages_for(item),
        reasoning_effort='high',
        seed=42
    )
    return response.choices[0].message.content

In [ ]:
evaluate(gpt_5__1, test)